# 02 · Votação de features

**Features F3 e F4** (critérios 3 e 4) · donos: ver `config/equipe.yaml` · **status: F3 e F4 prontas; o registro no MLflow é adicionado pela F6**

> Cada seção chama o módulo da própria feature (`src/features.py` e `src/votacao.py`).

| | |
|---|---|
| **Entradas** | `ciclo_atual.json`, `dev.parquet`, `cfg['selecao']` |
| **Saídas** | `data/processed/features_eleitas.json` · `evidencias/selecao/rankings_<cycle_id>.csv` · `evidencias/votacao/votos_<cycle_id>.csv` · registro no MLflow |

**Regras do enunciado que se aplicam**
- Três métodos no mesmo notebook (SelectKBest, importância do Random Forest e RFE, ou três modelos com feature importance).
- Regra de votação definida **antes** (está em `config.yaml`); registrar tabela de votos, desempate e mínimo de features.
- Ajustar só no treino; com validação cruzada, dentro de cada fold.

Contrato completo na docstring do módulo `src/` correspondente.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))

from src.config import carregar_config, carregar_contexto
from src.preprocessamento import carregar_particao, construir_preprocessador, lista_features

cfg = carregar_config()
ctx = carregar_contexto(cfg, RAIZ)  # escrito pelo notebook 01
print(f"cycle_id={ctx['cycle_id']} | data_version={ctx['data_version']}")

## Seção F3 · três métodos de seleção (`src/features.py`)

In [ ]:
from IPython.display import display

from src.features import calcular_rankings_por_fold, listas_aprovadas, salvar_rankings, tabela_rankings

dev = carregar_particao("dev", cfg, RAIZ)
X_bruto = dev[lista_features(cfg)]
y = dev[ctx["colunas"]["alvo"]]
tempo = dev[ctx["colunas"]["tempo"]]
k_top = cfg["selecao"]["k_top"]
print(f"dev: {len(dev):,} linhas | {X_bruto.shape[1]} colunas antes do pré-processador | k_top={k_top}")
print("métodos:", cfg["selecao"]["metodos"])

# Pré-processador e métodos são reajustados só com o treino de cada fold.
rankings, por_fold = calcular_rankings_por_fold(X_bruto, y, tempo, cfg)
tabela = tabela_rankings(rankings, por_fold, k_top)
print(f"{len(tabela)} features após o pré-processador (união dos folds)")

### Top-k de cada método
Score médio entre os folds. As escalas diferem entre métodos, então só a ordem dentro de cada coluna é comparável. Se houver empate na fronteira, esta tabela corta em ordem alfabética; o conjunto completo entra na concordância abaixo.

### Lista de features aprovadas por método
Cada método aprova as `k_top` melhores. Empatadas na fronteira entram todas, e é por essas listas que a votação conta os votos.

In [ ]:
aprovadas = listas_aprovadas(rankings, k_top)
for metodo, lista in aprovadas.items():
    print(f"{metodo} ({len(lista)}): {lista}\n")

In [ ]:
lado_a_lado = {}
for metodo, score in rankings.items():
    top = score.sort_values(ascending=False, kind="mergesort").head(k_top)
    lado_a_lado[(metodo, "feature")] = top.index.to_list()
    lado_a_lado[(metodo, "score")] = top.round(4).to_list()
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)
display(pd.DataFrame(lado_a_lado, index=range(1, len(top) + 1)))

### Tabela completa de scores por método
Mesmo índice nos três métodos. `posicao` 1 é a melhor; `folds_presente` diz em quantos folds o one-hot gerou a coluna.

In [ ]:
display(tabela.sort_values("posicao_random_forest").round(4))

### Concordância entre os métodos e estabilidade entre folds
Os métodos só agregam na votação se não forem cópias um do outro, e um ranking que muda muito entre folds merece desconfiança. Features empatadas na fronteira entram todas no top-k, que pode ficar maior que k.

In [ ]:
tops = {m: set(lista) for m, lista in aprovadas.items()}
print("empates na fronteira (tamanho do top-k):", {m: len(t) for m, t in tops.items() if len(t) > k_top})
display(pd.DataFrame(
    {a: {b: len(tops[a] & tops[b]) for b in tops} for a in tops}
).rename_axis("features em comum no top-k"))

estabilidade = {}
for metodo, grupo in por_fold.groupby("metodo"):
    tops_fold = [set(g.nlargest(k_top, "score", keep="all")["feature"]) for _, g in grupo.groupby("fold")]
    estabilidade[metodo] = {
        f"folds {i + 1}-{i + 2}": len(a & b) / len(a | b) for i, (a, b) in enumerate(zip(tops_fold, tops_fold[1:]))
    }
display(pd.DataFrame(estabilidade).T.round(2).rename_axis("Jaccard do top-k entre folds seguidos"))

em_algum_top = set.union(*tops.values())
parciais = tabela.loc[sorted(em_algum_top), "folds_presente"]
print("no top-k de algum método, mas ausentes em algum fold:", parciais[parciais < cfg["avaliacao"]["cv"]["n_splits"]].to_dict())

In [ ]:
caminhos_rankings = salvar_rankings(tabela, por_fold, ctx, RAIZ)
for nome, caminho in caminhos_rankings.items():
    print(f"{nome}: {caminho.relative_to(RAIZ)}")

### Limitações da seleção
- **Cada método tem um viés.** A informação mútua é univariada e ignora redundância. A importância por impureza favorece variáveis contínuas e de muitos valores. O RFE depende dos coeficientes de um modelo linear, e dummies não têm a mesma escala das numéricas padronizadas. É por isso que a votação da F4 combina os três.
- **Atributos redundantes dividem o crédito.** `tem_agente` e `agent_SEM_AGENTE` são exatamente complementares, e `total_noites` é a soma de outras duas colunas. O RFE pode manter uma e descartar a outra sem que a descartada seja inútil.
- **Média entre folds não é seleção por fold.** O treino dos folds finais inclui a validação dos anteriores, então o CV da F5 com um único conjunto eleito sai levemente otimista. Só a validação do último fold e o teste ficam intocados pela seleção. `rankings_por_fold_<cycle_id>.csv` guarda cada fold, caso a F5 queira medir esse efeito.
- **`country` foi retirada antes desta seleção.** O artigo do dataset avisa que a nacionalidade correta costuma ser conhecida só no check-in. Pelo mesmo aviso, `adults`, `children`, `stays_in_*` e `meal` podem carregar parte do problema; ficam como suposição documentada na Etapa 1.

## Seção F4 · votação e eleição (`src/votacao.py`)

**Regra, definida em `config.yaml` antes de rodar:** cada método vota nas features que aprovou, e é eleita a feature com pelo menos 2 dos 3 votos. Dois votos exigem que dois critérios diferentes concordem, sem depender da aprovação unânime de métodos que têm vieses diferentes. Se sobrarem menos de 10 eleitas, o conjunto é completado pelo melhor ranking médio. Empates são desfeitos por mais votos, depois menor ranking médio, depois nome.

In [ ]:
from src.votacao import eleger_features, salvar_votacao, votar

regra = cfg["selecao"]["votacao"]
print("regra:", regra, "| k_top:", k_top)

tabela_votos = votar(rankings, cfg)
eleitas = eleger_features(tabela_votos, cfg)
print(f"{len(eleitas)} features eleitas de {len(tabela_votos)}")
print("distribuição de votos:", tabela_votos["votos"].value_counts().sort_index(ascending=False).to_dict())

colunas_voto = [f"voto_{m}" for m in rankings] + ["votos", "ranking_medio"]
display(tabela_votos.assign(eleita=tabela_votos.index.isin(eleitas))[colunas_voto + ["eleita"]].query("votos > 0").round(2))

In [ ]:
n_folds = cfg["avaliacao"]["cv"]["n_splits"]
parciais = tabela.loc[eleitas, "folds_presente"]
print("eleitas ausentes em algum fold (a F5 precisa tratar):", parciais[parciais < n_folds].to_dict())

caminhos_votacao = salvar_votacao(tabela_votos, eleitas, ctx, cfg, RAIZ)
for nome, caminho in caminhos_votacao.items():
    print(f"{nome}: {caminho.relative_to(RAIZ)}")
print("eleitas:", eleitas)

### Limitações da votação
- **Registro no MLflow:** o enunciado pede votos, conjunto final, parâmetros e `cycle_id` no MLflow. Essa parte é da F6, que a adiciona ao fim deste notebook.
- **Votos valem igual:** o RFE é o método menos estável e pesa tanto quanto os outros. Por isso a tabela guarda o ranking médio e os votos de cada método.
- **Um único conjunto por ciclo:** vem da média entre folds, então o CV da F5 tende a sair levemente otimista.